## Getting Data Into Snowflake
Data only creates value once it's IN Snowflake. The cost of getting ingestion wrong: brittle pipelines, surprise credit bills, and analysts blocked waiting on data. This lab shows how to pick the right load path and stand it up.
> **Edition:** Snowpipe auto-ingest and CoCo-assisted migration are walkthrough-only because they require external infrastructure or source-system prerequisites.

In this hands-on lab you'll work through 5 sections:

| # | Section | Outcome |
|---|---------|---------|
| 1 | Choosing a Load Path | match the source and arrival pattern to the right ingestion service |
| 2 | Staging and File Formats | inspect files and define reusable parsing rules for CSV, JSON, and Parquet |
| 3 | Batch Load with COPY INTO | create a table from detected columns, load files, and handle a newly added column |
| 4 | Continuous Load with Snowpipe (auto-ingest) | load new files from cloud event notifications without managing a warehouse |
| 5 | CoCo-Assisted Legacy Migration | follow the documented six-stage guided workflow for a first legacy workload |

### Setup

Create the lab database and tag this session.

In [ ]:
-- Attribution: tag this session's queries (no privilege needed; role-agnostic)
ALTER SESSION SET QUERY_TAG = '{"origin":"sf_ace","name":"getting-data-into-snowflake","version":{"major":1,"minor":0},"attributes":{"program":"ace-webinar-series","webinar":"w03","source":"sql"}}';

USE ROLE SYSADMIN;

CREATE WAREHOUSE IF NOT EXISTS GETTING_DATA_INTO_SNOWFLAKE_WH
  WAREHOUSE_SIZE = XSMALL
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Query warehouse for the Getting Data Into Snowflake hands-on lab';
USE WAREHOUSE GETTING_DATA_INTO_SNOWFLAKE_WH;

CREATE DATABASE IF NOT EXISTS GETTING_DATA_INTO_SNOWFLAKE_HOL;
USE DATABASE GETTING_DATA_INTO_SNOWFLAKE_HOL;
USE SCHEMA PUBLIC;

## Section 1 — Choosing a Load Path

**The problem:** A pipeline becomes expensive or brittle when its ingestion service does not match how the source produces data.


Start with the source and arrival pattern. Use [COPY INTO](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table)
for file batches, [Snowpipe](https://docs.snowflake.com/en/user-guide/data-load-snowpipe-intro)
for continuously arriving files, Snowpipe Streaming for rows, and [Openflow](https://docs.snowflake.com/en/user-guide/data-integration/openflow/about)
or an existing partner connector
for supported databases and software-as-a-service sources.


> **Outcome:** match the source and arrival pattern to the right ingestion service.

## Choose from the source outward

The useful first question is not who owns an export schedule. It is what the source emits and how quickly
Snowflake needs to receive it.

| Arrival pattern | Start with | Why |
|---|---|---|
| Files loaded on demand or on a schedule | **COPY INTO** | You control when a warehouse loads the batch. |
| New files arriving continuously | **Snowpipe** | Cloud events queue new files for serverless loading. |
| Rows from applications, event buses, or streaming systems | **Snowpipe Streaming** | Rows are written without first creating staged files. |
| Supported databases or software-as-a-service sources | **Openflow** | Snowflake-managed connectors and runtimes handle extraction and ingestion. |
| A source already standardized on a partner tool | **Fivetran / partner connector** | Reuse the operating model that is already working. |

[CDC](https://docs.snowflake.com/en/user-guide/data-integration/openflow/about) = Change Data Capture, which
replicates inserts, updates, and deletes from a source database.
Openflow supports database CDC, software-as-a-service ingestion, streaming services, unstructured sources,
and custom processors. Partner connectors remain valid when a customer already uses one or when the required
source is not covered by the available Openflow connectors.

> Docs: [Data loading overview](https://docs.snowflake.com/en/user-guide/data-load-overview) · [About Openflow](https://docs.snowflake.com/en/user-guide/data-integration/openflow/about)

## Decision tree

```text
Does the source produce FILES?
  YES -> Do files arrive continuously through cloud events?
            YES -> Snowpipe
            NO  -> COPY INTO
  NO  -> Does the source produce ROWS or EVENTS?
            YES -> Snowpipe Streaming
            NO  -> Is it a supported database or SaaS source?
                      YES -> Openflow
                      NO  -> Openflow custom flow or a partner connector
```

Existing partner standards still matter. If [Fivetran or another partner connector](https://docs.snowflake.com/en/user-guide/ecosystem-etl)
is already deployed and governed, it can remain in place while the team evaluates Openflow for new sources.
Replacing an established connector is a separate architecture decision.

For this lab, the source produces files. We use **COPY INTO** for the live batch and then walk through the
**Snowpipe** version for files that arrive continuously.

> Docs: [Snowpipe Streaming](https://docs.snowflake.com/en/user-guide/snowpipe-streaming/data-load-snowpipe-streaming-overview) · [Openflow connectors](https://docs.snowflake.com/en/user-guide/data-integration/openflow/connectors/about)

## Where Openflow fits

```text
Database / SaaS / API
          |
          v
Openflow connector + runtime
          |
          v
     Snowflake table
```

Openflow is a first-class path when a supported connector matches the source. A runtime executes the connector
flow, while the connector configuration defines the source and destination. Database connectors can use CDC;
software-as-a-service connectors ingest records exposed by the source API.

Connector deployment and source networking need more setup than this self-contained SQL lab. The webinar keeps
the choice and architecture in the main story and provides a separate demonstration script for the setup flow.

> Docs: [About Openflow](https://docs.snowflake.com/en/user-guide/data-integration/openflow/about) · [Openflow connector overview](https://docs.snowflake.com/en/user-guide/data-integration/openflow/connectors/about)

## Section 2 — Staging and File Formats

**The problem:** A load cannot be trusted until you know which files are present and how Snowflake will interpret their fields.


A [stage](https://docs.snowflake.com/en/user-guide/data-load-overview#stages) points to files, while a
[file format](https://docs.snowflake.com/en/sql-reference/sql/create-file-format) records the parsing rules.
Inspect the stage first, then infer a schema before creating the target table.


> **Outcome:** inspect files and define reusable parsing rules for CSV, JSON, and Parquet.

## Stages and formats answer two different questions

A stage answers **where are the files?** A file format answers **how should Snowflake parse them?**
This lab uses a public Amazon S3 stage for the Tasty Bytes batch and an internal stage for a small,
deterministic schema-evolution demonstration.

Use [CSV](https://docs.snowflake.com/en/sql-reference/sql/create-file-format#type-csv) for flat exports,
[JSON](https://docs.snowflake.com/en/sql-reference/sql/create-file-format#type-json) for nested records, and
[Parquet](https://docs.snowflake.com/en/sql-reference/sql/create-file-format#type-parquet) for columnar analytics data.

| Format | Typical fit | Parsing concern |
|---|---|---|
| **CSV** | Flat exports with broad tool support | Headers, delimiters, quoting, and null values |
| **JSON** | Nested application or API records | Arrays and semi-structured values |
| **Parquet** | Columnar analytics data | Logical types and source schema consistency |

> Docs: [Data loading overview](https://docs.snowflake.com/en/user-guide/data-load-overview) · [CREATE FILE FORMAT](https://docs.snowflake.com/en/sql-reference/sql/create-file-format)

In [ ]:
-- Step 2.1: Create one external stage, one internal stage, and reusable formats.
USE ROLE SYSADMIN;
USE DATABASE GETTING_DATA_INTO_SNOWFLAKE_HOL;
USE SCHEMA PUBLIC;

CREATE OR REPLACE STAGE GDIS_STAGE
  URL = 's3://sfquickstarts/tastybytes/'
  COMMENT = 'Public Tasty Bytes sample data';

CREATE OR REPLACE STAGE GDIS_SCHEMA_STAGE
  COMMENT = 'Internal stage for the schema evolution demonstration';

CREATE OR REPLACE FILE FORMAT GDIS_FF_CSV
  TYPE = CSV
  FIELD_DELIMITER = ','
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  NULL_IF = ('NULL', 'null', '')
  TRIM_SPACE = TRUE
  ERROR_ON_COLUMN_COUNT_MISMATCH = TRUE;

CREATE OR REPLACE FILE FORMAT GDIS_FF_CSV_HEADER
  TYPE = CSV
  FIELD_DELIMITER = ','
  PARSE_HEADER = TRUE
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  NULL_IF = ('NULL', 'null', '')
  TRIM_SPACE = TRUE
  ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

CREATE OR REPLACE FILE FORMAT GDIS_FF_JSON
  TYPE = JSON
  STRIP_OUTER_ARRAY = TRUE;

CREATE OR REPLACE FILE FORMAT GDIS_FF_PARQUET
  TYPE = PARQUET
  USE_LOGICAL_TYPE = TRUE;

SHOW FILE FORMATS LIKE 'GDIS_FF%';

### Inspect the bucket before loading

`LIST` returns each staged file's name, compressed size, checksum, and last-modified timestamp. Narrow the path
or add a pattern so a large bucket does not bury the files you intend to load.

**You should see:** compressed CSV files under `raw_pos/menu/`.

> Docs: [LIST](https://docs.snowflake.com/en/sql-reference/sql/list)

In [ ]:
-- Step 2.2: Inspect the exact folder used by the batch load.
LIST @GDIS_STAGE/raw_pos/menu/ PATTERN = '.*[.]csv[.]gz';

## Section 3 — Batch Load with COPY INTO

**The problem:** Hand-written target tables drift away from incoming files and turn a routine load into a schema incident.


Use [INFER_SCHEMA](https://docs.snowflake.com/en/sql-reference/functions/infer_schema) with
`CREATE TABLE ... USING TEMPLATE`, then load by column name and enable automatic schema evolution for new columns.


> **Outcome:** create a table from detected columns, load files, and handle a newly added column.

## Create the target from the files

The first load uses the menu files inspected in Section 2. Their last field contains a
[VARIANT](https://docs.snowflake.com/en/sql-reference/data-types-semistructured#variant) JSON value with commas, so this
familiar batch uses an explicit landing-table contract. The smaller schema-change example later in the section
uses `INFER_SCHEMA` and `CREATE TABLE ... USING TEMPLATE` end to end.

> Docs: [CREATE TABLE USING TEMPLATE](https://docs.snowflake.com/en/sql-reference/sql/create-table) · [INFER_SCHEMA](https://docs.snowflake.com/en/sql-reference/functions/infer_schema)

In [ ]:
-- Step 3.1: Create the menu landing table with its explicit source contract.
USE ROLE SYSADMIN;
USE DATABASE GETTING_DATA_INTO_SNOWFLAKE_HOL;
USE SCHEMA PUBLIC;

CREATE OR REPLACE TABLE GDIS_MENU_TABLE (
  MENU_ID NUMBER,
  MENU_TYPE_ID NUMBER,
  MENU_TYPE VARCHAR,
  TRUCK_BRAND_NAME VARCHAR,
  MENU_ITEM_ID NUMBER,
  MENU_ITEM_NAME VARCHAR,
  ITEM_CATEGORY VARCHAR,
  ITEM_SUBCATEGORY VARCHAR,
  COST_OF_GOODS_USD NUMBER(38,4),
  SALE_PRICE_USD NUMBER(38,4),
  MENU_ITEM_HEALTH_METRICS_OBJ VARIANT
);

DESCRIBE TABLE GDIS_MENU_TABLE;

### Load using the explicit column order

This source uses the explicit column order defined above. The later schema-change example switches to
header-based matching, which is required for automatic schema evolution.

`ON_ERROR = 'CONTINUE'` keeps loading valid rows when an error is found. The COPY result reports parsed rows,
loaded rows, and detected errors for each file; the next cells inspect the table and validate the last load.

**You should see:** the menu file loaded, representative menu values, and no validation errors.

> Docs: [COPY INTO table](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table)

In [ ]:
-- Step 3.2: Load the menu files using the explicit table column order.
USE WAREHOUSE GETTING_DATA_INTO_SNOWFLAKE_WH;
COPY INTO GDIS_MENU_TABLE
  FROM @GDIS_STAGE/raw_pos/menu/
  FILE_FORMAT = (FORMAT_NAME = GDIS_FF_CSV)
  ON_ERROR = 'CONTINUE';

In [ ]:
-- Step 3.3: Confirm the row count and inspect representative values.
SELECT
    COUNT(*) OVER () AS TOTAL_ROWS,
    MENU_ITEM_NAME,
    ITEM_CATEGORY,
    SALE_PRICE_USD
  FROM GDIS_MENU_TABLE
  ORDER BY MENU_ITEM_ID
  LIMIT 10;

In [ ]:
-- Step 3.3b: Return all errors from the last standard COPY load in this session.
SELECT *
  FROM TABLE(VALIDATE(GDIS_MENU_TABLE, JOB_ID => '_last'));

### Re-run the unchanged batch

By default, `COPY INTO` skips unchanged staged files that were already loaded into the same target table.
`FORCE = TRUE` reloads them and can create duplicates.

**You should see:** zero newly loaded rows for the unchanged files.

> Docs: [COPY INTO load options](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table#copy-options-copyoptions)

In [ ]:
-- Step 3.4: Re-run the same load. Unchanged files are skipped.
COPY INTO GDIS_MENU_TABLE
  FROM @GDIS_STAGE/raw_pos/menu/
  FILE_FORMAT = (FORMAT_NAME = GDIS_FF_CSV);

### Build a repeatable schema-change example

The next cells write two small CSV files to the internal stage. The first has three columns. The second adds
`LOYALTY_TIER`. This keeps the demonstration deterministic while using the same file-loading path a cloud bucket
would use.

> Docs: [Unloading into a stage](https://docs.snowflake.com/en/user-guide/data-unload-snowflake) · [Automatic schema evolution](https://docs.snowflake.com/en/user-guide/data-load-schema-evolution)

In [ ]:
-- Step 3.5: Create version 1 and version 2 files with a deliberate new column.
REMOVE @GDIS_SCHEMA_STAGE PATTERN = 'schema_demo/.*';

COPY INTO @GDIS_SCHEMA_STAGE/schema_demo/v1/customers.csv
  FROM (
    SELECT column1 AS CUSTOMER_ID, column2 AS CUSTOMER_NAME, column3 AS REGION
      FROM VALUES
        (1, 'Ada', 'WEST'),
        (2, 'Grace', 'EAST')
  )
  FILE_FORMAT = (TYPE = CSV FIELD_OPTIONALLY_ENCLOSED_BY = '"' COMPRESSION = NONE)
  HEADER = TRUE
  SINGLE = TRUE
  OVERWRITE = TRUE;

COPY INTO @GDIS_SCHEMA_STAGE/schema_demo/v2/customers.csv
  FROM (
    SELECT column1 AS CUSTOMER_ID, column2 AS CUSTOMER_NAME, column3 AS REGION,
           column4 AS LOYALTY_TIER
      FROM VALUES
        (3, 'Katherine', 'CENTRAL', 'GOLD'),
        (4, 'Dorothy', 'WEST', 'SILVER')
  )
  FILE_FORMAT = (TYPE = CSV FIELD_OPTIONALLY_ENCLOSED_BY = '"' COMPRESSION = NONE)
  HEADER = TRUE
  SINGLE = TRUE
  OVERWRITE = TRUE;

LIST @GDIS_SCHEMA_STAGE/schema_demo/;

In [ ]:
-- Step 3.6: Inspect the detected version 1 columns before creating a table.
SELECT COLUMN_NAME, TYPE, NULLABLE, ORDER_ID
  FROM TABLE(
    INFER_SCHEMA(
      LOCATION => '@GDIS_SCHEMA_STAGE/schema_demo/v1/',
      FILE_FORMAT => 'GDIS_FF_CSV_HEADER'
    )
  )
  ORDER BY ORDER_ID;

### Create the table from the reviewed schema

Pause on the inference output first. Confirm the column names, inferred types, nullability, and order before
creating the target. `USING TEMPLATE` consumes the same required metadata fields to build the table.

For automatic evolution on later CSV loads, keep all four requirements together: enable schema evolution on
the table, load with `MATCH_BY_COLUMN_NAME`, use a role with `EVOLVE SCHEMA` or `OWNERSHIP`, and set
`ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE` when the CSV format uses `PARSE_HEADER`.

> Docs: [INFER_SCHEMA](https://docs.snowflake.com/en/sql-reference/functions/infer_schema) · [Automatic schema evolution](https://docs.snowflake.com/en/user-guide/data-load-schema-evolution)

In [ ]:
-- Step 3.7: Create the table from the reviewed schema and enable evolution.
CREATE OR REPLACE TABLE GDIS_CUSTOMER_TABLE
  USING TEMPLATE (
    SELECT ARRAY_AGG(
             OBJECT_CONSTRUCT(
               'COLUMN_NAME', COLUMN_NAME,
               'TYPE', TYPE,
               'NULLABLE', NULLABLE,
               'ORDER_ID', ORDER_ID
             )
           ) WITHIN GROUP (ORDER BY ORDER_ID)
      FROM TABLE(
        INFER_SCHEMA(
          LOCATION => '@GDIS_SCHEMA_STAGE/schema_demo/v1/',
          FILE_FORMAT => 'GDIS_FF_CSV_HEADER'
        )
      )
  );

ALTER TABLE GDIS_CUSTOMER_TABLE SET ENABLE_SCHEMA_EVOLUTION = TRUE;

COPY INTO GDIS_CUSTOMER_TABLE
  FROM @GDIS_SCHEMA_STAGE/schema_demo/v1/
  FILE_FORMAT = (FORMAT_NAME = GDIS_FF_CSV_HEADER)
  MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

### Load the later file and inspect the new column

File-based schema evolution requires the table setting, `MATCH_BY_COLUMN_NAME`, and `EVOLVE SCHEMA` or
ownership on the target table. For CSV with `PARSE_HEADER`, the file format must also set
`ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE`. The lab uses the table owner, `SYSADMIN`.

**You should see:** `LOYALTY_TIER` added to the table and populated for the two rows from version 2.

> Docs: [Enable automatic table schema evolution](https://docs.snowflake.com/en/user-guide/data-load-schema-evolution)

In [ ]:
-- Step 3.8: Load a later file whose header adds LOYALTY_TIER.
COPY INTO GDIS_CUSTOMER_TABLE
  FROM @GDIS_SCHEMA_STAGE/schema_demo/v2/
  FILE_FORMAT = (FORMAT_NAME = GDIS_FF_CSV_HEADER)
  MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

DESCRIBE TABLE GDIS_CUSTOMER_TABLE;

In [ ]:
-- Step 3.9: Confirm old rows remain and new rows use the added column.
SELECT CUSTOMER_ID, CUSTOMER_NAME, REGION, LOYALTY_TIER
  FROM GDIS_CUSTOMER_TABLE
  ORDER BY CUSTOMER_ID;

### File arrival is not row-level change capture

`COPY INTO` and Snowpipe process files presented for loading. They do not inspect a file that was already loaded
and translate later edits inside that same file into row-level inserts, updates, or deletes. If the source system
changes individual records, use a CDC-capable database connector or another design that carries those operations.

Automatic schema evolution is also limited to supported file loads and Snowpipe. Ordinary `INSERT` statements
do not evolve the target table automatically.

> Docs: [Automatic schema evolution](https://docs.snowflake.com/en/user-guide/data-load-schema-evolution) · [About Openflow](https://docs.snowflake.com/en/user-guide/data-integration/openflow/about)

### Verify the tables in Snowsight (read-only)

[Open in Snowsight](https://app.snowflake.com/_deeplink/#/data/databases/GETTING_DATA_INTO_SNOWFLAKE_HOL)

Read-only — confirm what you just built; all objects were created by the SQL above.

> **Apply on your account**
>
> In production, grant the loading role EVOLVE SCHEMA on the target table, preserve a review process for unexpected
> source changes, and monitor copy history. Automatic schema evolution adds columns during supported file loads;
> it does not interpret row-level edits inside a file that was already processed.

## Section 4 — Continuous Load with Snowpipe (auto-ingest)

> **Outcome:** load new files from cloud event notifications without managing a warehouse.

## Continuous Load with Snowpipe (auto-ingest)

> **Walkthrough only:** auto-ingest requires S3 event notifications on a
> bucket you own; the public Tasty Bytes bucket can't emit them. Use these
> cells as a reference when you wire up your own bucket.

**Snowpipe** uses Snowflake-provided compute to load files queued by cloud event notifications. For Amazon S3,
an object-created event notifies a Snowflake-managed Amazon Simple Queue Service (SQS) queue. The notification
identifies the file; it does not contain the file data.

> **Snowpipe vs Snowpipe Streaming:** this section is *file-based* Snowpipe
> (a file lands → COPY runs). If your source emits **rows** instead of files
> (Kafka, IoT, or application events), use **Snowpipe Streaming**. It is Snowflake's real-time, row-based
> ingestion service and complements rather than replaces file-based Snowpipe.

### How it works (auto-ingest path)

```
S3 bucket  ──(new file)──▶  S3 event notification
                                  │
                                  ▼
                            SQS queue (owned by Snowflake)
                                  │  notification_channel ARN
                                  ▼
                            Snowpipe serverless worker
                                  │  COPY INTO <table>
                                  ▼
                            Target Snowflake table
```

| Concept | Detail |
|---|---|
| **Serverless** | No warehouse to manage or size; Snowflake provisions compute on demand |
| **Event-driven** | S3 object-created events notify the Snowflake-managed SQS queue |
| **Load tracking** | Use pipe status and copy history to monitor file loads |
| **`notification_channel`** | The SQS ARN you wire into AWS after creating the pipe |

**RBAC Pattern 10 note:** `SYSADMIN` creates the pipe (data-layer object).
The creating role needs the documented privileges on the database, schema, stage, and target table.

> **Docs:**
> - [CREATE PIPE](https://docs.snowflake.com/en/sql-reference/sql/create-pipe)
> - [Automating Snowpipe for Amazon S3](https://docs.snowflake.com/en/user-guide/data-load-snowpipe-auto-s3)
> - [Automate continuous data loading](https://docs.snowflake.com/en/user-guide/data-load-snowpipe-auto)

In [ ]:
/* WALKTHROUGH ONLY — this cell is reference SQL.
   AUTO_INGEST=TRUE requires S3 event notifications on a bucket you own.
   The public Tasty Bytes bucket (sfquickstarts) cannot emit S3 events to
   a Snowflake-managed SQS queue. Run this against your own private bucket.
   Source: https://docs.snowflake.com/en/sql-reference/sql/create-pipe
           https://docs.snowflake.com/en/user-guide/data-load-snowpipe-auto-s3
*/

-- Step 1: Switch to the data-owning role (RBAC Pattern 10).
USE ROLE SYSADMIN;
USE DATABASE GETTING_DATA_INTO_SNOWFLAKE_HOL;
USE SCHEMA PUBLIC;

-- Step 2: Create the pipe with auto-ingest enabled.
--   AUTO_INGEST = TRUE  → Snowpipe listens on an SQS queue for S3 ObjectCreated events.
--   The COPY INTO body mirrors the batch section but runs serverlessly on each new file.
CREATE OR REPLACE PIPE GDIS_PIPE
  AUTO_INGEST = TRUE
  COMMENT    = 'Continuous load from GDIS_STAGE into GDIS_MENU_TABLE via S3 events'
  AS
    COPY INTO GDIS_MENU_TABLE
      FROM   @GDIS_STAGE/raw_pos/menu/
      FILE_FORMAT = (FORMAT_NAME = GDIS_FF_CSV)  -- reuse the named CSV format from prior section
      ON_ERROR    = 'CONTINUE';

-- Step 3: After CREATE PIPE succeeds, retrieve the SQS ARN you need in AWS.
--   The `notification_channel` column contains the ARN — copy it to the S3 console.
SHOW PIPES LIKE 'GDIS_PIPE';

-- Alternatively, query pipe status (includes pendingFileCount, executionState):
--   SELECT SYSTEM$PIPE_STATUS('GDIS_PIPE');

/* END WALKTHROUGH */

## Apply on Your Own Bucket

After you configure secure access to your private bucket, create the external stage and pipe. Then wire the
S3 event notification to the Snowflake-managed SQS queue:

### Step A — Get the SQS ARN

```sql
SHOW PIPES LIKE 'MY_PIPE';
```

Copy the value in the **`notification_channel`** column — that is the
Snowflake-managed SQS ARN for your account's region.

### Step B — Configure the S3 event notification

In the **AWS S3 Console → your bucket → Properties → Event notifications**:

| Field | Value |
|---|---|
| **Name** | e.g. `snowpipe-auto-ingest` |
| **Event types** | `s3:ObjectCreated:*` (all PUT / POST / Copy / MPU) |
| **Destination** | SQS Queue |
| **SQS queue ARN** | Paste the `notification_channel` ARN from Step A |

> **Tip:** Scope the notification to a specific prefix (e.g. `raw_pos/menu/`)
> to reduce cost, event noise, and latency from unrelated uploads.
> AWS docs: [Enabling Amazon S3 event notifications](https://docs.aws.amazon.com/AmazonS3/latest/userguide/enable-event-notifications.html)

### Step C — Backfill files that arrived before the notification was live

If Snowpipe missed eligible files, use a manual refresh as a short-term recovery step. It is not a polling
mechanism or a regular ingestion schedule. It checks both the pipe and target-table load histories and queues
files that were staged within the previous seven days and were not already loaded by the same pipe or COPY INTO:

```sql
-- Queue eligible files missed during setup or an incident.
-- Source: https://docs.snowflake.com/en/sql-reference/sql/alter-pipe
ALTER PIPE GDIS_PIPE REFRESH;
```

After the refresh, monitor load progress:

```sql
SELECT SYSTEM$PIPE_STATUS('GDIS_PIPE');

-- Or query copy history for GDIS_MENU_TABLE:
SELECT *
  FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
       TABLE_NAME  => 'GDIS_MENU_TABLE',
       START_TIME  => DATEADD('hour', -1, CURRENT_TIMESTAMP())));
```

> **Docs:**
> - [ALTER PIPE](https://docs.snowflake.com/en/sql-reference/sql/alter-pipe)
> - [COPY_HISTORY](https://docs.snowflake.com/en/sql-reference/functions/copy_history)
> - [SYSTEM$PIPE_STATUS](https://docs.snowflake.com/en/sql-reference/functions/system_pipe_status)

## Section 5 — CoCo-Assisted Legacy Migration

**The problem:** Legacy migrations stall when teams treat thousands of SQL objects, source dependencies, data movement, and validation as separate manual projects.


The [Snowflake AIM Agent for Data Warehouses](https://docs.snowflake.com/en/migrations/aim-for-datawarehouses/overview)
is bundled with Snowflake CoCo CLI and guides an end-to-end migration workflow. It invokes SnowConvert for
deterministic source-code translation before AI-assisted remediation.


> **Outcome:** follow the documented six-stage guided workflow for a first legacy workload.

## Start with the guided migration workflow

> **Walkthrough only:** this workflow runs in Snowflake CoCo CLI, not inside the notebook. Confirm the source
> connection and migration prerequisites before planning a customer migration around it.

The [Snowflake AIM Agent for Data Warehouses](https://docs.snowflake.com/en/migrations/aim-for-datawarehouses/overview)
keeps project state across sessions and guides the documented six-stage sequence:

```text
Customer goal
     |
     v
CoCo CLI + /migrate
     |
     v
Connect -> Init -> Register -> Convert -> Assess -> Migrate
```

The Migrate stage includes dependency-aware deployment, data migration, testing, validation, and iterative fixes.

[SnowConvert AI](https://docs.snowflake.com/en/migrations/aim-for-datawarehouses/manual-migration/README) performs
deterministic source-code translation underneath the guided workflow. The current
SnowConvert CLI also includes source-specific data migration and validation workflows, so do not treat it as
code conversion only.

> Docs: [Snowflake AIM Agent for Data Warehouses](https://docs.snowflake.com/en/migrations/aim-for-datawarehouses/overview)

### What the assistant handles

- installs SnowConvert, `uv`, required Python packages, and applicable ODBC drivers on first run;
- registers source code and invokes deterministic conversion;
- analyzes conversion issues and identifies work that needs review;
- guides the Migrate stage, including deployment, data migration, testing, validation, and fixes.

Start in natural language or invoke the bundled migration skill directly:

```text
/migrate Assess our Oracle warehouse and convert the first finance workload.
```

Supported source connections for the guided workflow include SQL Server, Amazon Redshift, Oracle, Teradata,
and PostgreSQL. Deterministic code-conversion coverage includes additional source dialects, and capability
coverage varies by source.

> Docs: [AIM overview](https://docs.snowflake.com/en/migrations/aim-for-datawarehouses/overview) · [Troubleshooting and manual path](https://docs.snowflake.com/en/migrations/aim-for-datawarehouses/troubleshooting)

### Advanced reference: direct SnowConvert CLI

Advanced users can operate the SnowConvert AI command-line interface directly. The current code-only sequence is:

```bash
scai init my-project -l Oracle
cd my-project
scai code add -i /path/to/sql/files
scai code convert
scai code deploy
```

Use `scai --help` and command-specific help as the live syntax reference. Review the generated reports and
validate converted objects before deployment. Do not promise a fixed effort estimate from conversion results;
use the assessment to identify translated objects and remaining migration work.

> Docs: [Manual migration](https://docs.snowflake.com/en/migrations/aim-for-datawarehouses/manual-migration/README) · [SnowConvert CLI command reference](https://docs.snowflake.com/en/migrations/aim-for-datawarehouses/manual-migration/SCAI_Command_Reference)

## Cleanup

Drop all objects created during the lab.

In [ ]:
USE ROLE SYSADMIN;
USE DATABASE GETTING_DATA_INTO_SNOWFLAKE_HOL;
DROP PIPE IF EXISTS PUBLIC.GDIS_PIPE;
DROP DATABASE IF EXISTS GETTING_DATA_INTO_SNOWFLAKE_HOL;
DROP WAREHOUSE IF EXISTS GETTING_DATA_INTO_SNOWFLAKE_WH;